<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-light.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification 🚀 notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. We hope this notebook helps you get the most out of it — browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support!</div>

# Image Classification with WeightsLab

This notebook trains a small CNN on **MNIST** and instruments it with WeightsLab so every training signal is traced **back to the exact samples** producing it.

The idea is simple: most data problems (mislabels, outliers, class imbalance) stay invisible until your model tells you through the loss. By wrapping your training objects with `wl.watch_or_edit(...)`, WeightsLab records **per-sample** loss and metrics live — so you can rank the worst samples, spot bad labels, and curate the dataset **without restarting training**.

### What you'll do
1. Install WeightsLab.
2. Wrap the model, optimizer, dataloaders, loss, and metric with the SDK.
3. Train for a few hundred steps while per-sample signals are captured.
4. Inspect the highest-loss samples right in the notebook — and see how to open the live **Weights Studio** UI.

## Setup

Install WeightsLab from PyPI. On Colab, the free **T4 GPU** runtime is plenty for this demo (`Runtime → Change runtime type → T4 GPU`).

In [1]:
%pip install -q weightslab
!pip install --upgrade "protobuf>=6.31.1"
%pip install torchvision>=0.16

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
  Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl (327 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
weightslab 1.3.2 requires protobuf<7,>

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [2]:
import os
import tempfile
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.examples.utils.baseline_models.pytorch.models import FashionCNN as CNN
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

08/07/2026-13:18:01.517 INFO:root:setup_logging: WeightsLab logging initialized - Log file: /tmp/tmpclib5yqf/weightslab_logs/weightslab_20260708_131801.log
08/07/2026-13:18:01.518 INFO:weightslab:<module>: WeightsLab package initialized - Log level: INFO, Log to file: True
08/07/2026-13:18:01.520 INFO:weightslab:<module>: 
╭─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                     │
│  $$\      $$\           $$\           $$\         $$\               $$\                 $$\         │
│  $$ | $\  $$ |          \__|          $$ |        $$ |              $$ |                $$ |        │
│  $$ |$$$\ $$ | $$$$$$\  $$\  $$$$$$\  $$$$$$$\  $$$$$$\    $$$$$$$\ $$ |       $$$$$$\  $$$$$$$\    │
│  $$ $$ $$\$$ |$$  __$$\ $$ |$$  __$$\ $$  __$$\ \_$$  _|  $$  _____|$$ |       \____$$\ $$  __$$\   │
│  $$$$  _$$$$ |$$$$$$$$ |$$ |$$ /  $$ |$$ |  $$ | 

## 2. A dataset that carries sample identity

WeightsLab attributes every signal to a **sample id**. This thin wrapper around torchvision's MNIST returns `(image, id, label)` and attaches a virtual filepath per sample, so signals can later be traced back to individual images in the UI.

In [3]:
class MNISTCustomDataset(Dataset):
    """MNIST that returns (image, index, label) and tracks a virtual filepath."""

    def __init__(self, root, train=True, download=False, transform=None, max_samples=None):
        self.mnist = datasets.MNIST(root=root, train=train, download=download, transform=None)
        self.transform = transform
        self.train = train
        self.max_samples = max_samples
        split = "train" if train else "test"
        self.filepaths = {}
        for idx in range(len(self.mnist)):
            if max_samples is not None and idx >= max_samples:
                break
            label = self.mnist.targets[idx].item()
            self.filepaths[idx] = os.path.join(
                "MNIST", "processed", split, f"class_{label}", f"sample_{idx:05d}.pt"
            )

    def __len__(self):
        if self.max_samples is not None:
            return min(len(self.mnist), self.max_samples)
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.transform:
            image = self.transform(image)
        return image, idx, label

## 3. Hyperparameters

We keep the run short so the notebook finishes quickly. Wrapping the config with `flag="hyperparameters"` lets the Studio UI read (and live-edit) these values while training.

In [4]:
log_dir = tempfile.mkdtemp(prefix="weightslab_mnist_")

parameters = {
    "experiment_name": "mnist_classification",
    "device": str(device),
    "training_steps_to_do": 2000,         # raise for a longer live session
    "eval_full_to_train_steps_ratio": 100, # full eval every 100 steps
    "write_export_ratio": 100,             # export signals every 100 steps
    "root_log_dir": log_dir,
    "serving_grpc": True,                  # expose gRPC so Studio can connect
}

wl.watch_or_edit(parameters, flag="hyperparameters", poll_interval=1.0)
print("Experiment logs ->", log_dir)

08/07/2026-13:18:01.577 INFO:weightslab.src:watch_or_edit: LoggerQueue for experiment history has been initialized and registered.
08/07/2026-13:18:01.580 INFO:weightslab.components.checkpoint_manager:__init__: CheckpointManager initialized at /tmp/weightslab_mnist_crjdgwx4
08/07/2026-13:18:01.581 INFO:weightslab.src:watch_or_edit: Registered new checkpoint manager in ledger
Experiment logs -> /tmp/weightslab_mnist_crjdgwx4


## 4. Wrap the training objects

This is the heart of WeightsLab. Each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave exactly like the originals, but now report their state and per-sample signals to WeightsLab.

In [5]:
data_root = os.path.join(log_dir, "data")
os.makedirs(data_root, exist_ok=True)

train_ds = MNISTCustomDataset(root=data_root, train=True, download=True,
                              transform=transforms.ToTensor(), max_samples=6000)
test_ds = MNISTCustomDataset(root=data_root, train=False, download=True,
                             transform=transforms.ToTensor(), max_samples=2000)

# Model + optimizer
model = wl.watch_or_edit(CNN().to(device), flag="model", device=device)
optimizer = wl.watch_or_edit(optim.Adam(model.parameters(), lr=0.01), flag="optimizer")

# Tracked dataloaders
train_loader = wl.watch_or_edit(
    train_ds, flag="data", loader_name="train_loader",
    batch_size=16, shuffle=True, is_training=True,
    compute_hash=False, preload_labels=True,
    preload_metadata=False, enable_h5_persistence=True,
)
test_loader = wl.watch_or_edit(
    test_ds, flag="data", loader_name="test_loader",
    batch_size=128, shuffle=False, is_training=False,
    compute_hash=False, preload_labels=True,
    preload_metadata=False, enable_h5_persistence=True,
)

# Watched losses + metric (they log themselves per sample)
train_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                   flag="loss", signal_name="train-loss-CE", log=True)
test_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                  flag="loss", signal_name="test-loss-CE", log=True)
metric = wl.watch_or_edit(Accuracy(task="multiclass", num_classes=10).to(device),
                          flag="metric", signal_name="metric-ACC", log=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 22.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 587kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 5.84MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]


08/07/2026-13:18:05.362 INFO:weightslab.backend.model_interface:__init__: Using checkpoint manager from ledger
08/07/2026-13:18:05.373 INFO:weightslab.data.data_samples_with_ops:__init__: [DataSampleTrackingWrapper] H5 persistence enabled at /tmp/weightslab_mnist_crjdgwx4/checkpoints/data/data.h5
08/07/2026-13:18:05.392 INFO:weightslab.data.data_samples_with_ops:__init__: Preloading sample statistics for PHYSICAL indices in split 'train_loader' with: preload_labels=True,
08/07/2026-13:18:05.393 INFO:weightslab.data.data_samples_with_ops:__init__: Metadata will be loaded on demand from the wrapped dataset for split 'train_loader' when accessed, which may increase latency on first access but reduces initialization time.


Initializing ledger for split 'train_loader': 100%|██████████| 6000/6000 [00:01<00:00, 3193.40it/s]

08/07/2026-13:18:07.295 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] Registering split 'train_loader' with 6000 samples.


08/07/2026-13:18:07.439 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] After annotation expansion: 6000 samples → 6000 annotation rows.
08/07/2026-13:18:07.440 INFO:weightslab.data.h5_array_store:__init__: [H5ArrayStore] Initialized with cache limit: 2048MB
08/07/2026-13:18:07.550 INFO:weightslab.data.data_samples_with_ops:__init__: [DataSampleTrackingWrapper] H5 persistence enabled at /tmp/weightslab_mnist_crjdgwx4/checkpoints/data/data.h5
08/07/2026-13:18:07.564 INFO:weightslab.data.data_samples_with_ops:__init__: Preloading sample statistics for PHYSICAL indices in split 'test_loader' with: preload_labels=True,
08/07/2026-13:18:07.567 INFO:weightslab.data.data_samples_with_ops:__init__: Metadata will be loaded on demand from the wrapped dataset for split 'test_loader' when accessed, which may increase latency on first access but reduces initialization time.


Initializing ledger for split 'test_loader': 100%|██████████| 2000/2000 [00:00<00:00, 3449.45it/s]

08/07/2026-13:18:08.158 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] Registering split 'test_loader' with 2000 samples.
08/07/2026-13:18:08.187 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] After annotation expansion: 2000 samples → 2000 annotation rows.


## 5. Train & evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so the loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [6]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)

        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total_loss = loss.mean()
        total_loss.backward()
        optimizer.step()
    return total_loss.detach().cpu().item()


def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)

            loss = criterion(logits, labels, batch_ids=ids, preds=preds)
            losses += loss.mean()
            metric.update(logits, labels)

            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(
                preds_raw=logits, targets=labels, batch_ids=ids, preds=preds,
                signals={
                    "test_metric/Accuracy_per_sample": correct,
                    "test_metric/Inverse_Accuracy_per_sample": 1.0 - correct,
                },
            )
    return (losses / n_batches).item(), (metric.compute() * 100).item()

## 6. (Optional) Expose the backend for the live UI

Skip this section if you only want the inline results above. To watch training **live in Weights Studio on your own machine**, expose this notebook's gRPC port (`50051`) with a **raw-TCP** tunnel.

We use [`bore`](https://github.com/ekzhang/bore) — a tiny open-source TCP tunnel with a free public relay (`bore.pub`). **No signup, no account, no credit card.**

> ⚠️ It must be a **raw-TCP** tunnel — gRPC's HTTP/2 frames have to pass through untouched. (This is also why we avoid `ngrok`, which now requires a credit card for TCP endpoints.)
>
> 🔓 `bore.pub` is a shared public relay: the random port is the only thing protecting your endpoint. Fine for a demo — don't stream sensitive data through it.

In [14]:
# `bore` — zero-signup raw-TCP tunnel (https://github.com/ekzhang/bore)
import os, re, tarfile, threading, urllib.request, subprocess

BORE = "v0.6.0"
if not os.path.exists("bore"):
    urllib.request.urlretrieve(
        f"https://github.com/ekzhang/bore/releases/download/{BORE}/bore-{BORE}-x86_64-unknown-linux-musl.tar.gz",
        "bore.tar.gz",
    )
    with tarfile.open("bore.tar.gz") as t:
        t.extractall()
    os.chmod("bore", 0o755)

proc = subprocess.Popen(
    ["./bore", "local", "50051", "--to", "bore.pub"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

endpoint = None
for line in proc.stdout:
    print(line, end="")
    m = re.search(r"bore\.pub:(\d+)", line)
    if m:
        endpoint = f"bore.pub:{m.group(1)}"
        break

# Keep draining bore's output so its pipe never fills and stalls the tunnel.
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

print("\n" + "=" * 60)
print("Backend exposed at:", endpoint)
print("On your OWN machine (docker deamon running), run these two commands:")
print("    weightslab ui launch")
print(f"    weightslab tunnel {endpoint}")
print("=" * 60)

2026-07-08T13:30:32.515154Z  INFO bore_cli::client: connected to server remote_port=29504
2026-07-08T13:30:32.515182Z  INFO bore_cli::client: listening at bore.pub:29504

Backend exposed at: bore.pub:29504
On your OWN machine (docker deamon running), run these two commands:
    weightslab ui launch
    weightslab tunnel bore.pub:29504


## 7. Serve & train

`wl.serve(serving_grpc=True)` starts the background gRPC server (non-blocking) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop, periodically evaluating and exporting signals.

Leave this cell **running** while you watch it stream in the UI.

In [15]:
# Need to be separate from the main training loop
wl.serve(serving_grpc=parameters["serving_grpc"])

08/07/2026-13:30:39.541 INFO:weightslab.trainer.trainer_services:_run_security_preflight: 

# #######################################
# #######################################
[gRPC] Security preflight checks:
	TLS: DISABLED (unencrypted traffic)
	Auth tokens: NONE configured
	! WARNING: GRPC_TLS_ENABLED=0. Traffic will be unencrypted. Use only for development.
	! WARNING: No GRPC_AUTH_TOKEN/GRPC_AUTH_TOKENS configured. Only transport-level trust (TLS/mTLS) will protect RPC access.
# #######################################
# #######################################

08/07/2026-13:30:39.545 INFO:weightslab.trainer.trainer_services:grpc_serve: [gRPC] Watchdogs disabled via WEIGHTSLAB_DISABLE_WATCHDOGS.
08/07/2026-13:30:39.547 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Thread callback started
08/07/2026-13:30:39.548 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Creating ThreadPoolExecutor with 6 worker threads (n_workers_grpc=None, m

[PreviewCache]:   0%|          | 0/2000 [00:00<?, ?sample/s]

08/07/2026-13:30:39.734 WARNING:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] TLS disabled; using insecure transport on 0.0.0.0:50051
08/07/2026-13:30:39.742 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Port 50051 bound successfully.
08/07/2026-13:30:39.745 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Server started and listening on 0.0.0.0:50051


[PreviewCache]:  41%|████      | 811/2000 [00:01<00:01, 667.45sample/s]

08/07/2026-13:30:40.850 INFO:weightslab.data.dataframe_manager:flush_if_needed_nonblocking: Checking if flush to H5 is needed (non-blocking). Pending count: 0.
08/07/2026-13:30:40.852 INFO:weightslab.data.dataframe_manager:flush_if_needed_nonblocking: Completed non-blocking flush check. Pending count after flush: 0.


[PreviewCache]:  93%|█████████▎| 1854/2000 [00:02<00:00, 621.15sample/s]

08/07/2026-13:30:42.571 INFO:weightslab.components.global_monitoring:resume: 
Attempting to resume training...
08/07/2026-13:30:42.612 INFO:weightslab.components.global_monitoring:resume: Hashes by module: ['1535fb08', '9b3b6c5e', 'fff89066']
08/07/2026-13:30:42.614 INFO:weightslab.components.global_monitoring:resume: Resuming training now...
08/07/2026-13:30:42.616 INFO:weightslab.components.global_monitoring:resume: Hashes by module on resume: ['1535fb08', '9b3b6c5e', 'fff89066']
08/07/2026-13:30:42.619 INFO:weightslab.components.global_monitoring:resume: 
Training resumed as modules hashes have been computed: ['1535fb08', '9b3b6c5e', 'fff89066'].


In [16]:
wl.start_training(timeout=3)

steps = parameters["training_steps_to_do"]
eval_ratio = parameters["eval_full_to_train_steps_ratio"]
export_ratio = parameters["write_export_ratio"]
n_test_batches = len(test_loader)

test_loss = test_acc = None
from tqdm.notebook import tqdm
pbar = tqdm(range(steps), desc="Training")
for step in pbar:
    age = model.get_age() if hasattr(model, "get_age") else step

    train_loss = train(train_loader, model, optimizer, train_criterion, device)

    if age > 0 and age % eval_ratio == 0:
        test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)

    if age > 0 and age % export_ratio == 0:
        wl.write_history()
        wl.write_dataframe()

    postfix = {"loss": f"{train_loss:.3f}"}
    if test_acc is not None:
        postfix["test_acc"] = f"{test_acc:.1f}%"
    pbar.set_postfix(postfix)

# Final export of signal history + per-sample dataframe
wl.write_history()
wl.write_dataframe()
print("Training complete. Logs at:", log_dir)

Training:   0%|          | 0/2000 [00:00<?, ?it/s]

08/07/2026-13:32:00.072 INFO:weightslab.data.dataframe_manager:flush: Flushing 100 buffered records to DataFrame.
08/07/2026-13:32:00.100 INFO:weightslab.data.dataframe_manager:flush: Applied 100 buffered records to DataFrame.
08/07/2026-13:32:00.101 INFO:weightslab.data.dataframe_manager:flush: Checking if flush to H5 is needed. Pending count: 100.
08/07/2026-13:32:00.667 INFO:weightslab.data.dataframe_manager:flush: Completed flush. Pending count after flush: 0.
08/07/2026-13:32:00.769 INFO:weightslab.data.dataframe_manager:flush: Flushing 100 buffered records to DataFrame.
08/07/2026-13:32:00.796 INFO:weightslab.data.dataframe_manager:flush: Applied 100 buffered records to DataFrame.
08/07/2026-13:32:00.797 INFO:weightslab.data.dataframe_manager:flush: Checking if flush to H5 is needed. Pending count: 100.
08/07/2026-13:32:01.047 INFO:weightslab.data.dataframe_manager:flush: Completed flush. Pending count after flush: 0.
08/07/2026-13:32:01.149 INFO:weightslab.data.dataframe_manager

## 8. Which samples hurt the most?

WeightsLab exports a per-sample dataframe to `root_log_dir`. Ranking by loss surfaces the samples the model struggles with — the usual suspects for **mislabels and outliers**. This is the same signal the Studio UI visualizes; here we peek at it inline.

In [23]:
import glob
import pandas as pd

paths = glob.glob(os.path.join(log_dir, "**", "*.json"), recursive=True)
json_files = sorted(paths,
              key=os.path.getmtime)
if json_files:
    df = pd.read_json(json_files[-1])
    print(f"Loaded {json_files[-1]}  (shape={df.shape})")
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    if loss_cols:
        display(df.sort_values(loss_cols[-1], ascending=False).head(100))
    else:
        display(df.head())
else:
    print("No dataframe export found — run the training cell first.")

Loaded /tmp/weightslab_mnist_crjdgwx4/d8e01c87_dataframe.json  (shape=(8000, 16))


,sample_id,annotation_id,discarded,group_id,last_seen,member_rank,origin,prediction,prediction_raw,signals//test-loss-CE,signals//test_metric/Accuracy_per_sample,signals//test_metric/Inverse_Accuracy_per_sample,signals//train-loss-CE,tag:ToVerify,target,task_type
0,0,0,False,0,3609,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,True,[5],
3867,3867,0,False,3867,3488,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[9],
3876,3876,0,False,3876,3282,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[0],
3875,3875,0,False,3875,3005,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[5],
3874,3874,0,False,3874,3405,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[9],
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3972,3972,0,False,3972,3017,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[5],
3971,3971,0,False,3971,3116,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[1],
3970,3970,0,False,3970,2433,0,train_loader,7,NaN,NaN,NaN,NaN,2.46115,False,9,
3968,3968,0,False,3968,3652,0,train_loader,[7],NaN,NaN,NaN,NaN,2.46115,False,[8],


## 9. See it live in Weights Studio

Everything above ran headless. The real payoff is the **Weights Studio** UI, where you browse the highest-loss images, filter by class, and curate the dataset mid-training.

Studio runs as a local Docker stack (a static frontend + an Envoy gRPC-Web proxy). **Colab has no Docker daemon**, so you don't run the UI *inside* Colab — you run Studio on your own machine and point it at this notebook's backend using the endpoint printed in **Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab

# Terminal 1 — launch the UI (plaintext HTTP, the default)
weightslab ui launch                       # opens http://localhost:5173

# Terminal 2 — bridge the Colab backend to localhost:50051
weightslab tunnel bore.pub:12345           # <-- the host:port from Section 6
```

Then open **http://localhost:5173** — Studio connects through your local Envoy → `weightslab tunnel` → `bore.pub` → this Colab backend, and you watch training stream live.

> ℹ️ How it fits together: the UI's Envoy proxy already dials `localhost:50051`; `weightslab tunnel` simply makes the remote Colab backend appear there. It must stay **plaintext** end-to-end (the default `weightslab ui launch`, raw-TCP `bore`) so gRPC's HTTP/2 frames pass through untouched.

Prefer to keep it all local? Run this same example on your own machine (`weightslab start example --cls`) and launch the UI next to it — no tunnel required.

---

<div align="center">
Crafted with 💙 by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> — if WeightsLab helps you catch a bad label, drop us a ⭐ on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>